In [ ]:
# import stuff
import asymmetree.treeevolve as te
import networkx as nx
import matplotlib.pyplot as plt
import random
import numpy as np
import MISC_Functions

from bmg_Tony import bmg
from bmg_fun import convert_to_nx
from insert_hybrid import insertHybrid
from all_BCEA import BCEA1, BCEA2, BCEA3, BCEA4

In [ ]:
# ======================================================================================
#  Baumgenerierung - generiert einen zufälligen Genbaum
# ======================================================================================

def generateGeneTree(nSpecies: int, max_leaves: int):

    while True:

        # species tree
        speciesTree = te.species_tree_n_age(
            age = 1.0,
            n = nSpecies
            #, contraction_probability=0.0, contraction_proportion=0.2, contraction_bias="exponential"
        )
        # gene tree
        T = te.dated_gene_tree(
            speciesTree, dupl_rate=1.0, loss_rate=0, hgt_rate=0.2, gc_rate=0.2, prohibit_extinction="per_species", dupl_polytomy=0.5
        )
        # prune all loss branches and the planted root
        geneTree = te.prune_losses(T)

        nLeaves = len(list(geneTree.leaves()))
        if nLeaves < max_leaves and nLeaves > 2:  #wenn es nur zwei Blätter (oder nSpezies?????) gibt, so lassen sich keine Hybriden einfügen
            break

    Tree = convert_to_nx(geneTree)

    # print('Baum generiert mit',nSpecies, 'Spezies und', nLeaves, 'Blättern')

    return Tree

# ======================================================================================
#  Build BCN     -    HIER NEUE BCEA ALGORITHMEN EINFÜGEN
# ======================================================================================
def build_BCN(BMG :nx.DiGraph, mode :str, model :str) ->nx.DiGraph:

    if model == "BC":
        BCN = BCEA1(BMG)
    elif model == "BCR" :
        BCN = BCEA2(BMG)
    elif model == "BCEA":
        BCN = BCEA3(BMG,mode)
    elif model == "MLBCEA":
        BCN = BCEA4(BMG)
    else:
        raise ValueError("Kein gültiges Model als Input - Zum testen eines neuen Modells, dieses in build_BCN funktion einfügen")

    return BCN

# =====================================================================
# Kriterien-Pruefung   weak vs. strict
# =====================================================================


def check_criteria_weak(BMG: nx.DiGraph):
    """
    Prueft, ob BMG mindestens eine "Kette" fehlender/asymmetrischer Best
    Matches enthaelt, die der (einstufige) BIC-Cherry-Expansion-Algorithmus
    (BCEA) -- mit reziprozitaets-bevorzugter Kandidatenwahl -- unter der
    weak-Definition nicht korrekt darstellen kann.
 
    Formale Kriterium
    ------------------
    Mit N(x) = {y : (x,y) in E(BMG)} (echte Best Matches von x) und
    L_c = {v in V : sigma(v) = c} (alle Gene der Farbe c):
 
    BMG ist NICHT durch den (einstufigen) BCEA darstellbar genau dann, wenn
 
        exists x in V, exists Farbe c != sigma(x):
            N(x) ∩ L_c  ⊊  L_c                     (x deckt c nicht vollstaendig ab)
            und  fuer alle z in N(x) ∩ L_c:  (z,x) not in E    (KEIN Kandidat reziprok)
 
    Kurz: x braucht eine Korrektur bzgl. Farbe c (mind. ein Gen dieser
    Farbe fehlt x als Match), UND unter x's tatsaechlichen Matches dieser
    Farbe ist KEIN EINZIGER reziprok. Weil die Kandidatenwahl im BCEA
    reziproke Kandidaten bevorzugt (siehe insertNode/findCandidate), reicht
    schon EIN reziproker Kandidat aus, um den Bruch zu vermeiden -- daher
    "fuer alle" statt (wie in einer frueheren Version) "es existiert einer".
 
    Deckt x eine Farbe schon vollstaendig ab, bleibt die urspruengliche
    P-Cherry ohnehin unangetastet -- daher zaehlt das nicht als Bruchgrund,
    selbst wenn dort asymmetrische Matches vorkommen.
 
    Parameters
    ----------
    BMG : networkx.DiGraph
        Das zu pruefende (Ziel-)BMG. Jeder Knoten braucht ein "color"-Attribut.
 
    Returns
    -------
    has_chain : bool
        True, wenn mindestens eine Kette gefunden wurde.
    chains : list[dict]
        Alle gefundenen Ketten, je mit den Schluesseln "x", "color",
        "candidates" (ALLE tatsaechlichen Matches von x dieser Farbe --
        jeder davon ist nicht-reziprok, sonst waere keine Kette gemeldet),
        "missing_example" (ein Beispiel-y der Farbe, das x fehlt).
    """
    nodes_by_color: dict = {}
    for n, data in BMG.nodes(data=True):
        nodes_by_color.setdefault(data["color"], set()).add(n)
 
    chains = []
 
    for x in BMG.nodes():
        color_x = BMG.nodes[x]["color"]
        successors_x = set(BMG.successors(x))
 
        for c, members in nodes_by_color.items():
            if c == color_x:
                continue
 
            matched = successors_x & members
            missing = members - matched
            if not missing:
                continue  # x deckt Farbe c vollstaendig ab -> keine Korrektur noetig
 
            reciprocal = [z for z in matched if BMG.has_edge(z, x)]
            if reciprocal:
                continue  # BCEA waehlt bevorzugt einen reziproken Kandidaten -> kein Bruch erwartet
 
            # alle Kandidaten dieser Farbe sind nicht-reziprok -> Bruch erwartet
            chains.append({
                "x": x, "color": c,
                "candidates": sorted(matched, key=str),
                "missing_example": next(iter(missing)),
            })
 
    return len(chains) > 0, chains


def check_criteria_strong(BMG: nx.DiGraph):
    """
    Prueft auf das strong-spezifische Bruchkriterium -- aber nur den
    Fall, in dem ein Bruch GARANTIERT/ERZWUNGEN ist: ein doppelt zu
    korrigierender P-Knoten p_{x1,y1}, bei dem JEDE moegliche Kombination
    aus Kandidat x2 (fuer y1) und Kandidat y2 (fuer x1) bereits eine echte
    Beziehung zueinander hat. Gibt es mindestens EIN "sicheres" Paar (ohne
    jede Kante zwischen x2 und y2), kann eine entsprechend angepasste
    Kandidatenwahl (siehe choose_correction_pair unten) den Bruch
    vermeiden -- das zaehlt daher NICHT als erzwungener Bruch.
 
    Parameters
    ----------
    BMG : networkx.DiGraph
        Das zu pruefende Ziel-BMG (i.d.R. mode="strong").
 
    Returns
    -------
    has_forced_break : bool
    breaks : list[dict]
        Nur fuer P-Knoten OHNE jedes sichere Kandidatenpaar. Enthaelt
        "x1", "y1", "unsafe_pairs" (ALLE moeglichen (x2,y2)-Kombinationen,
        jede davon zerstoert bei Wahl ein echtes Match -- welches genau,
        ist der Kandidatenwahl im BCEA ueberlassen).
    """
    breaks = []
    nodes = list(BMG.nodes())
 
    for x1, y1 in ((a, b) for a in nodes for b in nodes if a < b):
        color_x1 = BMG.nodes[x1]["color"]
        color_y1 = BMG.nodes[y1]["color"]
        if color_x1 == color_y1:
            continue
        if BMG.has_edge(x1, y1) or BMG.has_edge(y1, x1):
            continue  # p_x1y1 wird nicht beidseitig korrigiert

        candidates_x2 = [
                x2 for x2 in BMG.nodes
                if BMG.nodes[x2]["color"] == color_x1
                and x2 != x1
            ]

        candidates_y2 = [
                y2 for y2 in BMG.nodes
                if BMG.nodes[y2]["color"] == color_y1
                and y2 != y1
            ]

        if not candidates_x2 or not candidates_y2:
            continue
 
        safe_pairs = [
            (x2, y2) for x2 in candidates_x2 for y2 in candidates_y2
            if not BMG.has_edge(x2, y2) and not BMG.has_edge(y2, x2)
        ]
        if safe_pairs:
            continue  # Bruch vermeidbar -> zaehlt nicht als ERZWUNGENER Bruch
 
        # KEIN sicheres Paar -> jede Wahl zerstoert irgendein echtes Match
        unsafe_pairs = [(x2, y2) for x2 in candidates_x2 for y2 in candidates_y2]
        breaks.append({"x1": x1, "y1": y1, "unsafe_pairs": unsafe_pairs})
 
    return len(breaks) > 0, breaks


# =====================================================================
# Haupt-Loop mit Ereignis-Sammlung
# =====================================================================

def run_experiment(nTrees: int, speciesRange, maxLeaves: int, nRuns: int, model :str, mode: str, maxHybrids: int):
    """
    Generiert nTrees unterschiedliche Bäume mit nSpecies = random.choice(speciesrange) Spezien und maximal maxLeaves Blättern 
    Auf jeden Baum werden nRuns runs ausgeführt.
    Jeder run fügt schrittweise Hybriden ein, bis entweder die BMGS (bmg(Network) und bmg(BCN)) nicht mehr 
    übereinstimmen oder die Grenze maxHybrids erreicht wird.

    Nach jedem hinzugefügten Hybriden wird überprüft, ob die BMGs noch übereinstimmen und geschaut, ob dies
    unseren Erwartungen entspricht (check_criteria strong vs. weak)
    Das Ergebnis wird in den entsprechenden Listen gespeichert und kann später betrachtet werden (z.B. plot_anomalies).


    Returns
    -------
    events : list[dict]
        Ein Eintrag PRO Hybrid-Insertion (ueber alle Runs).
    anomalies : list[dict]
        False negatives der ALTEN globalen Kriteriums-Pruefung: broke=True,
        aber matched=False (siehe check_hybrid_criteria).
    false_positives : list[dict]
        matched=True, aber broke=False (alte Kriteriums-Pruefung).
    hypothesis_mismatches : list[dict]
        Faelle, in denen predicted_break != broke -- das ist der direkte
        Test eurer neuen Cherry-Hypothese. Idealerweise leer.
    hybrids_before_break : list[int]
        Anzahl Hybriden bis zum Bruch, NUR fuer Runs mit echtem Bruch
        (zensierte Runs, die das Limit erreicht haben, sind ausgeschlossen).
    """

    events = []
    anomalies = []
    false_positives = []
    hypothesis_mismatches = []
    hybrids_before_break = []

    if mode == 'strong' and model == 'MLBCEA':
        raise ValueError('the multilayered BCEA is not suitable for the strict best match definition - change MODE or MODEL')

    for t in range(nTrees):
        nSpecies = random.choice(speciesRange)
        Tree = generateGeneTree(nSpecies,maxLeaves)

        for i in range(nRuns):

            Network = Tree.copy()

            BMG = bmg(Tree, mode=mode)
            BCN = build_BCN(BMG, mode = mode, model = model)
            BMGBC = bmg(BCN, mode=mode)

            count = 0
            broke = False

            while nx.utils.graphs_equal(BMG, BMGBC) and count < maxHybrids:

                Network = insertHybrid(Network)

                BMG = bmg(Network, mode=mode)
                BCN = build_BCN(BMG, mode = mode, model = model)
                BMGBC = bmg(BCN, mode=mode)

                count += 1
                broke = not nx.utils.graphs_equal(BMG, BMGBC)

                if mode == 'weak':
                    predicted_break, fault = check_criteria_weak(BMG)

                if mode == 'strong':
                    predicted_break, fault = check_criteria_strong(BMG)


                events.append({
                    "tree": t,
                    "run": i, "hybrid_index": count,
                    "broke": broke,
                    "predicted_break": predicted_break,
                    "fault": fault
                })

                if broke and not predicted_break:
                    anomalies.append({
                        "tree": t,
                        "run": i, "hybrid_index": count,
                        "fault": fault,
                        "network": Network.copy(),
                    })

                if predicted_break and not broke:
                    false_positives.append({
                        "tree": t,
                        "run": i, "hybrid_index": count,
                        "fault": fault,
                        "broke": broke,
                        "network": Network.copy(),
                    })

                if predicted_break != broke:
                    hypothesis_mismatches.append({
                        "tree": t,
                        "run": i, "hybrid_index": count,
                        "predicted_break": predicted_break,
                        "broke": broke,
                        "fault": fault,
                        "network": Network.copy(), "BC": BCN.copy(),
                    })

            if broke:
                hybrids_before_break.append(count)
            # sonst: Run hat das Limit erreicht, ohne zu brechen -> zensiert, nicht mitzaehlen

    return events, anomalies, false_positives, hypothesis_mismatches, hybrids_before_break


def print_summary(events, hybrids_before_break, runs, mode, model,nTrees):
    tp = sum(e["broke"] and e["predicted_break"] for e in events)
    fn = sum(e["broke"] and not e["predicted_break"] for e in events)
    fp = sum((not e["broke"]) and e["predicted_break"] for e in events)
    tn = sum((not e["broke"]) and not e["predicted_break"] for e in events)

    print("=== Zusammenfassung Kriterium vs. BIC Cherry Expansion Algorithmus ===")
    print(f"=== Best match Definition: {mode:^6}                Algorithmus:{model:^6}===")
    print(f"TP (Hypothese sagt Break richtig vorher):           {tp}")
    print(f"FN (Break, aber Kriterium sagt diesn nicht vorher): {fn}   <- Algorithmus nicht gut genug, oder Kriterium zu strikt")
    print(f"FP (Kriterium sagt break hervor, aber kein break):  {fp}   <- Algorithmus funktioniert besser, oder Kriterium ist ")
    print(f"TN (kein break, break auch nicht vorhergesagt):     {tn}")
    if tp + fn:
        print(f"Recall:    {tp/(tp+fn):.2%}")
    if tp + fp:
        print(f"Precision: {tp/(tp+fp):.2%}")
    if fn == 0 and fp == 0:
        print(">>> Hypothese trifft auf ALLE Ereignisse exakt zu (100% Recall & Precision).")
    print(f"\nEchte (nicht zensierte) Breaks:                     {len(hybrids_before_break)} / {runs*nTrees}")
    if hybrids_before_break:
        print(f"Durchnittliche Anzahl der Hybriden bis Break:       {np.mean(hybrids_before_break):.2f}")


# =====================================================================
# 4. Plot-Funktionen    müssen noch überarbeitet werden
# =====================================================================

def get_color_map(*graphs):
    """Erzeugt eine konsistente Farbzuordnung fuer beliebige color-Attributwerte.
    Sortiert nach str(), damit gemischte Typen (z.B. int-Blattfarben und
    tuple-Hybridfarben) den Vergleich nicht zum Absturz bringen."""
    all_colors = set()
    for g in graphs:
        all_colors |= {data["color"] for _, data in g.nodes(data=True) if "color" in data}
    palette = plt.cm.tab10.colors
    return {c: palette[i % len(palette)] for i, c in enumerate(sorted(all_colors, key=str))}


def node_display_color(G: nx.DiGraph, n, color_map):
    """Blaetter bekommen ihre echte Speziesfarbe; interne/Hybrid-Knoten
    (die evtl. gar keine sinnvolle color haben, z.B. gemischte Tupel)
    werden neutral grau dargestellt."""
    if G.out_degree(n) == 0 and "color" in G.nodes[n]:
        return color_map[G.nodes[n]["color"]]
    return (0.8, 0.8, 0.8)


def layered_layout(G: nx.DiGraph):
    """Einfaches Ebenen-Layout fuer DAGs (Wurzel oben, Blaetter unten)."""
    pos = {}
    for depth, generation in enumerate(nx.topological_generations(G)):
        gen = list(generation)
        for j, node in enumerate(gen):
            pos[node] = (j - len(gen) / 2, -depth)
    return pos


def plot_network_and_bmg(Network: nx.DiGraph, BMG: nx.DiGraph, title: str = ""):
    color_map = get_color_map(Network, BMG)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Netzwerk (Ebenen-Layout, da DAG) ---
    pos_net = layered_layout(Network)
    node_colors_net = [node_display_color(Network, n, color_map) for n in Network.nodes()]
    nx.draw(
        Network, pos_net, ax=axes[0], with_labels=True,
        node_color=node_colors_net, node_size=600,
        font_size=8, arrows=True, arrowsize=12,
    )
    axes[0].set_title("Netzwerk")

    # --- BMG (kann Kanten in beide Richtungen haben -> spring_layout) ---
    pos_bmg = nx.spring_layout(BMG, seed=0)
    node_colors_bmg = [node_display_color(BMG, n, color_map) for n in BMG.nodes()]
    nx.draw(
        BMG, pos_bmg, ax=axes[1], with_labels=True,
        node_color=node_colors_bmg, node_size=600,
        font_size=8, arrows=True, arrowsize=12,
        connectionstyle="arc3,rad=0.15",  # trennt Hin-/Rueckkante optisch
    )
    axes[1].set_title("BMG")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_anomalies(anomalies, mode: str = "weak", limit=None):
    """Zeichnet Netzwerk + BMG fuer jede Anomalie (broke, aber Kriterium matcht nicht)."""
    for a in anomalies[:limit]:
        plot_network_and_bmg(
            a["network"],
            bmg(a["network"], mode=mode),
            title=(
                f"ANOMALIE - Run {a['run']}, Hybrid #{a['hybrid_index']}  |  "
                f"node1={a['node1']}, node2={a['node2']}\n"
                f"missing={a['missing']}  extra={a['extra']}"
            ),
        )


def plot_false_positives(false_positives, mode: str = "weak", limit=None):
    """Zeichnet Netzwerk + BMG fuer jeden False Positive (Kriterium matcht, aber kein Break)."""
    for fp in false_positives[:limit]:
        plot_network_and_bmg(
            fp["network"],
            bmg(fp["network"], mode=mode),
            title=(
                f"FALSE POSITIVE ({fp['label']}) - Run {fp['run']}, Hybrid #{fp['hybrid_index']}  |  "
                f"node1={fp['node1']}, node2={fp['node2']}"
            ),
        )


def plot_hypothesis_mismatches(hypothesis_mismatches, mode: str = "weak", limit=None):
    """Zeichnet Netzwerk + BC fuer jeden Fall, in dem die Cherry-Hypothese
    (asymmetrische Blatt-Cherry in BC <=> Break) NICHT zutraf."""
    for m in hypothesis_mismatches[:limit]:
        plot_network_and_bmg(
            m["BC"],
            bmg(m["network"], mode=mode),
            title=(
                f"HYPOTHESE-MISMATCH - Run {m['run']}, Hybrid #{m['hybrid_index']}  |  "
                f"predicted_break={m['predicted_break']}, broke={m['broke']}\n"
            ),
        )


In [81]:
# ======================================================================================
#  Konfiguration
# ======================================================================================

# random.seed(0)
# np.random.seed(0)
MODE = 'weak'
MODEL = 'MLBCEA'    # Entweder BC oder BCR oder BCEA oder MLBCEA

trees: int = 100                    # Anzahl Bäume, die generiert werden sollen
speciesRange = (2,10)               # Anzahl Spezies im Genbaum
max_leaves: int = 30                # Obergrenze Blätter (Rechenaufwand)
runs: int = 20                      # Runs (Umbau von Baum zu Netzwerk durch Hybride) pro Baum
max_hybrids: int = 20               # maximale Anzahl an Hybriden pro Baum


events, anomalies, false_positives, hypothesis_mismatches, hybrids_before_break = run_experiment(
    nTrees= trees,
    speciesRange= speciesRange,
    maxLeaves= max_leaves,
    nRuns= runs, 
    model= MODEL, 
    mode= MODE, 
    maxHybrids= max_hybrids
)


print_summary(events, hybrids_before_break, runs, MODE, MODEL, trees)

# # Beispiele anschauen (limit=None fuer alle):
# plot_false_positives(false_positives,mode = MODE, limit = 5)
# plot_hypothesis_mismatches(hypothesis_mismatches, mode=MODE, limit=5)
# plot_anomalies(anomalies, mode = MODE, limit = 5)


=== Zusammenfassung Kriterium vs. BIC Cherry Expansion Algorithmus ===
=== Best match Definition:  weak                 Algorithmus:MLBCEA===
TP (Hypothese sagt Break richtig vorher):           556
FN (Break, aber Kriterium sagt diesn nicht vorher): 0   <- Algorithmus nicht gut genug, oder Kriterium zu strikt
FP (Kriterium sagt break hervor, aber kein break):  18798   <- Algorithmus funktioniert besser, oder Kriterium ist 
TN (kein break, break auch nicht vorhergesagt):     14498
Recall:    100.00%
Precision: 2.87%

Echte (nicht zensierte) Breaks:                     556 / 2000
Durchnittliche Anzahl der Hybriden bis Break:       8.94
